# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, via its Croissant schema.

### Dataset Source
We use the published Croissant schema URL as the authoritative description and access point for dataset structure and reference.

In [ ]:
# Install mlcroissant if not already present
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the FAIR^2 Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
List available record sets and fields, referencing **all entity `@id` values**. This shows the structure according to the Croissant schema.

In [ ]:
# List all record sets and their fields, with @id references
record_sets_metadata = metadata.record_sets
if not record_sets_metadata:
    print("No record sets defined in the Croissant metadata.")
else:
    for rs in record_sets_metadata:
        print(f"Record Set: {rs['@id']} (name: {rs.get('name')}):")
        if 'fields' in rs:
            for field in rs['fields']:
                print(f"  Field: {field['@id']} (name: {field.get('name')}, dataType: {field.get('dataType')}, column: {field.get('column') if 'column' in field else 'N/A'})")
        else:
            print("  No fields listed in this record set.")

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame. Use valid record set and field `@id`s from the metadata above for reference.

**Example:**<br>
`record_set_id = '@id' of your chosen record set`

In [ ]:
# List all record set @ids
record_set_ids = [rs['@id'] for rs in metadata.record_sets] if metadata.record_sets else []
print("Available record set IDs:", record_set_ids)

# Choose a record set to extract records from (replace with valid @id)
if record_set_ids:
    chosen_record_set = record_set_ids[0]
    print(f"\nLoading records for record set: {chosen_record_set}")
    records_iter = dataset.records(record_set=chosen_record_set)
    records = list(records_iter)
    df = pd.DataFrame(records)
    print(f"First 5 records for {chosen_record_set}:")
    display(df.head())
    print("Field (column) names:", df.columns.tolist())
else:
    print("No record sets found; please check the Croissant metadata.")

## 4. Exploratory Data Analysis (EDA)
Let's process and inspect the data: filter, normalize, and group by attributes. _All fields and columns should be referenced by their `@id` as in the Croissant metadata._

In [ ]:
if record_set_ids:
    # Attempt to find numeric fields for basic EDA
    field_candidates = None
    for rs in metadata.record_sets:
        if rs['@id'] == chosen_record_set and 'fields' in rs:
            field_candidates = rs['fields']
            break

    numeric_field_id = None
    group_field_id = None
    # Prefer Float or Integer types
    if field_candidates:
        for f in field_candidates:
            # Use @id as required; attempt to find numeric field
            if isinstance(f.get('dataType'), str) and (
                'Float' in f['dataType'] or 'Integer' in f['dataType'] or 'Number' in f['dataType']):
                if f['@id'] in df.columns:
                    numeric_field_id = f['@id']
                    break
        for f in field_candidates:
            # Use a non-numeric field for grouping (e.g., 'description', 'sample', etc.)
            if f['@id'] in df.columns and f['@id'] != numeric_field_id:
                group_field_id = f['@id']
                break
    
    if numeric_field_id is None:
        print("Could not auto-detect a numeric field by @id. Please modify notebook to select one.")
    else:
        print(f"Selected numeric field (by @id): {numeric_field_id}")
        # Convert field to numeric (may contain non-numeric/NA)
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        # Drop NAs
        df_numeric = df.dropna(subset=[numeric_field_id])
        # Example: filter > median
        threshold = df_numeric[numeric_field_id].median()
        filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > median ({threshold}): {len(filtered_df)} rows")
        display(filtered_df.head())
        # Normalize
        filtered_df[numeric_field_id + "_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())
        
        # Group by group_field if found
        if group_field_id and group_field_id in df.columns:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped.head())
        else:
            print("No suitable categorical field found for grouping.")
else:
    print("No record set loaded; please load records to proceed.")

## 5. Visualization
Visualize numeric field distribution and relationships. You may adjust the numeric and group field (by `@id`) as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_numeric[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field is available, display boxplot
    if group_field_id and group_field_id in df_numeric.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_numeric)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization. Please run previous cells and ensure fields are correctly identified.")

## 6. Conclusion
In this notebook, we've:
- Loaded and inspected the Croissant metadata, referencing all entities by their canonical `@id`
- Explored record sets and their available fields
- Loaded tabular data, selected numeric and group fields by `@id`, and performed basic exploratory analysis
- Visualized value distributions and group statistics

This template can be extended for more detailed analyses by referencing additional record sets or linking fields, always using their unique Croissant `@id`s for portable, schema-compliant code.